# 08 — Insert a custom correction

This extension notebook keeps the same configure–run–inspect structure as the active research notebooks. It inserts one explicitly named experimental product after standard topo/BRDF correction and passes that product into the existing convolution stage without overwriting the canonical baseline.

## 1. Read the extension contract

Read `docs/reference/custom-correction-hook.md` before implementing a method. Work in an isolated output root, include the method and version in the filename, preserve ENVI metadata and NoData, process in chunks, and write provenance. Never replace the standard corrected pair.

In [ ]:
from pathlib import Path
from pprint import pprint

from spectralbridge.pipelines.pipeline import stage_convolve_all_sensors

RUN = False
base_folder = Path("outputs/custom_scs_v1")
product_code = "DP1.30006.001"
flight_stem = "NEON_D13_NIWO_DP1_L019-1_20230815_directional_reflectance"
method_name = "custom_scs"
method_version = "v1"
flight_dir = base_folder / flight_stem
standard_img = flight_dir / f"{flight_stem}_brdfandtopo_corrected_envi.img"
standard_hdr = standard_img.with_suffix(".hdr")
custom_stem = f"{flight_stem}_brdfandtopo_{method_name}_{method_version}_envi"
custom_img = flight_dir / f"{custom_stem}.img"
custom_hdr = flight_dir / f"{custom_stem}.hdr"
provenance_json = flight_dir / f"{custom_stem}_provenance.json"

## 2. Implement one scientifically reviewed transform

The placeholder is deliberately nonfunctional. Replace it with a chunked transform that writes a new `.img/.hdr` pair and provenance JSON. First validate an identity implementation before testing a controlled scientific change.

In [ ]:
def apply_custom_correction(
    input_img: Path,
    input_hdr: Path,
    output_img: Path,
    output_hdr: Path,
    output_provenance: Path,
) -> None:
    """Write a chunked ENVI transform and its provenance record."""
    raise NotImplementedError(
        "Implement and validate the correction; do not write an untracked copy."
    )

## 3. Run the custom hook, then reuse the standard downstream stage

The existence check prevents convolution from starting after an incomplete custom write. The downstream call is unchanged from notebook 03 except that its corrected input is the explicitly named experimental pair.

In [ ]:
result = None
if RUN:
    apply_custom_correction(
        standard_img,
        standard_hdr,
        custom_img,
        custom_hdr,
        provenance_json,
    )
    required_outputs = [custom_img, custom_hdr, provenance_json]
    missing = [path for path in required_outputs if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Custom correction did not write: {missing}")
    result = stage_convolve_all_sensors(
        base_folder=base_folder,
        product_code=product_code,
        flight_stem=flight_stem,
        corrected_img_path=custom_img,
        corrected_hdr_path=custom_hdr,
        resample_method="convolution",
        extraction_mode="full",
    )
    pprint(result)
else:
    print("Template only. Implement and test the correction before setting RUN = True.")

## 4. Check outputs and compare with the baseline

The custom pair and provenance must be present together. Use notebooks 04 and 05 on both baseline and experimental folders, then compare Landsat-space outputs rather than judging only the hyperspectral image.

In [ ]:
for label, path in {
    "standard image": standard_img,
    "custom image": custom_img,
    "custom header": custom_hdr,
    "provenance": provenance_json,
}.items():
    print(f"{label:>16}: exists={path.exists()} | {path}")

## 5. Required evidence

Record method name, version, parameters, source product, and software version. Test identity behavior, controlled synthetic change, NoData, metadata preservation, chunk equivalence, restart behavior, and baseline-versus-custom Landsat differences before proposing the correction as part of the package.